# Held-Out Cells e–h Extraction (generalization test, standalone)

**Purpose:** Extract everything needed to evaluate **frozen** policies (trained on cells a–d) on the four **held-out** ClusterData 2019 cells e–h — the workload-generalization experiment of the peer-review response (M2).

Produces, per cell x ∈ {e,f,g,h}:
1. `data/cells/cell_x.csv` — aggregate CPU demand (Dataset-1 filters, identical to a–d)
2. `data/cells/cell_x_tiers.csv` — measured per-tier split, deferrable = `priority ≤ 115` (free + beb per trace docs v3)
3. `data/machines/machines_x.csv` — fleet capacities
4. `data/power_model_eh.json` — per-cell power fits (PowerData2019 join, same method as cells a–d)
5. `data/jobs/durations_eh.json` — mean/median duration of FINISHed no-SLO collections (deadline model input)

**Standalone** — no other notebook needed. Grab the zip from the Colab **Files panel** (the download cell no-ops in the VS Code bridge).

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'aeee-thesis'

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)
print(f'Authenticated with project: {PROJECT_ID}')

In [ ]:
import os, json
import pandas as pd

os.makedirs('data/cells', exist_ok=True)
os.makedirs('data/machines', exist_ok=True)
os.makedirs('data/jobs', exist_ok=True)
CELLS = ['e', 'f', 'g', 'h']

for cell in CELLS:
    print(f'--- Cell {cell}: aggregate + tier curves ---')
    ds = f'`google.com:google-cluster-data`.clusterdata_2019_{cell}'
    query = f"""
    WITH cap AS (
        SELECT SUM(cpu_cap) AS cpu_capacity FROM (
            SELECT machine_id, MAX(capacity.cpus) AS cpu_cap
            FROM {ds}.machine_events GROUP BY 1
        )
    ),
    prio AS (
        SELECT collection_id, MAX(priority) AS priority
        FROM {ds}.collection_events WHERE type = 0 GROUP BY collection_id
    )
    SELECT
        CAST(FLOOR(u.start_time / (1e6 * 300)) AS INT64) AS time_bucket,
        SUM(u.average_usage.cpus) / (SELECT cpu_capacity FROM cap) AS total_norm,
        SUM(IF(p.priority <= 115, u.average_usage.cpus, 0))
            / (SELECT cpu_capacity FROM cap) AS batch_norm
    FROM {ds}.instance_usage u
    LEFT JOIN prio p USING (collection_id)
    WHERE (u.alloc_collection_id IS NULL OR u.alloc_collection_id = 0)
        AND (u.end_time - u.start_time) >= (5 * 60 * 1e6)
    GROUP BY 1 ORDER BY 1
    """
    df = client.query(query).to_dataframe()
    df['timestep'] = df['time_bucket'] - df['time_bucket'].min()
    df['cpu_demand_norm'] = df['total_norm'].clip(0.0, 1.0)
    df['batch_demand_norm'] = df['batch_norm'].clip(0.0, 1.0).clip(upper=df['cpu_demand_norm'])
    df['service_demand_norm'] = df['cpu_demand_norm'] - df['batch_demand_norm']
    df['batch_share'] = (df['batch_demand_norm'] / df['cpu_demand_norm'].clip(lower=1e-9)).clip(0, 1)
    df[['timestep', 'cpu_demand_norm']].to_csv(f'data/cells/cell_{cell}.csv', index=False)
    df[['timestep', 'cpu_demand_norm', 'batch_share', 'service_demand_norm',
        'batch_demand_norm']].to_csv(f'data/cells/cell_{cell}_tiers.csv', index=False)
    bf = df['batch_demand_norm'].sum() / df['cpu_demand_norm'].sum()
    print(f'  {len(df)} rows  batch_fraction={bf:.3f}  mean={df["cpu_demand_norm"].mean():.3f}')

print('Aggregate + tier curves done.')

In [ ]:
for cell in CELLS:
    q = f"""
    SELECT machine_id, MAX(capacity.cpus) AS cpu_capacity,
           MAX(capacity.memory) AS memory_capacity
    FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.machine_events
    GROUP BY 1
    HAVING cpu_capacity IS NOT NULL AND memory_capacity IS NOT NULL
    """
    df = client.query(q).to_dataframe()
    df['cell'] = cell
    df.to_csv(f'data/machines/machines_{cell}.csv', index=False)
    print(f'cell {cell}: {len(df)} machines, fleet CPU={df["cpu_capacity"].sum():.1f}')
print('Machines done.')

In [ ]:
import numpy as np

power_df = client.query("""
SELECT cell, CAST(FLOOR(time / (1e6*60*60)) AS INT64) AS hour_index,
       AVG(measured_power_util) AS avg_power_util
FROM `google.com:google-cluster-data`.`powerdata_2019.cell*`
WHERE NOT bad_measurement_data AND cell IN ('e','f','g','h')
GROUP BY 1, 2
""").to_dataframe()

per_cell = {}
for cell in CELLS:
    cap = float(client.query(f"""
        SELECT SUM(c) FROM (SELECT machine_id, MAX(capacity.cpus) c
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.machine_events
        GROUP BY 1)""").to_dataframe().iloc[0, 0])
    cpu = client.query(f"""
        SELECT CAST(FLOOR(start_time / (1e6*60*60)) AS INT64) AS hour_index,
               SUM(average_usage.cpus) / (12 * {cap}) AS avg_cpu_util
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.instance_usage
        WHERE (alloc_collection_id IS NULL OR alloc_collection_id = 0)
            AND (end_time - start_time) >= (5*60*1e6)
        GROUP BY 1""").to_dataframe()
    m = cpu.merge(power_df[power_df['cell'] == cell], on='hour_index')
    x, y = m['avg_cpu_util'].values, m['avg_power_util'].values
    ok = np.isfinite(x) & np.isfinite(y)
    A = np.column_stack([np.ones(ok.sum()), x[ok]])
    (a0, a1), *_ = np.linalg.lstsq(A, y[ok], rcond=None)
    pred = A @ [a0, a1]
    r2 = 1 - ((y[ok]-pred)**2).sum() / ((y[ok]-y[ok].mean())**2).sum()
    per_cell[cell] = {'idle_power': float(a0), 'slope': float(a1),
                      'peak_power': float(a0+a1), 'r_squared': float(r2), 'n': int(ok.sum())}
    print(f'cell {cell}: idle={a0:.3f} slope={a1:.3f} R2={r2:.3f} (n={ok.sum()})')

with open('data/power_model_eh.json', 'w') as f:
    json.dump({'per_cell_cpu_model': per_cell}, f, indent=2)
print('Power fits done -> data/power_model_eh.json')

In [ ]:
# Mean duration of FINISHed no-SLO collections (deadline-model input).
durs = {}
for cell in CELLS:
    q = f"""
    WITH s AS (SELECT collection_id, MIN(time) st, MAX(priority) pr
               FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.collection_events
               WHERE type = 0 GROUP BY 1),
         e AS (SELECT collection_id, MAX(time) et, MAX(type) tt
               FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.collection_events
               WHERE type IN (4,5,6) GROUP BY 1)
    SELECT AVG((e.et - s.st)/1e6) AS mean_dur, APPROX_QUANTILES((e.et - s.st)/1e6, 2)[OFFSET(1)] AS med_dur,
           COUNT(*) AS n
    FROM s JOIN e USING (collection_id)
    WHERE s.pr <= 115 AND e.tt = 6 AND e.et > s.st
    """
    r = client.query(q).to_dataframe().iloc[0]
    durs[cell] = {'mean_duration_sec': float(r['mean_dur']),
                  'median_duration_sec': float(r['med_dur']), 'n_completed': int(r['n'])}
    print(f'cell {cell}: mean={r["mean_dur"]:.0f}s median={r["med_dur"]:.0f}s n={r["n"]:,}')

with open('data/jobs/durations_eh.json', 'w') as f:
    json.dump(durs, f, indent=2)
print('Durations done -> data/jobs/durations_eh.json')

In [ ]:
import shutil
shutil.make_archive('cells_eh', 'zip', '.', 'data')
try:
    from google.colab import files
    files.download('cells_eh.zip')
except Exception as e:
    print(f'files.download unavailable ({e}); use the Files panel.')
print('Extract into C:\\Projects\\thesis\\ (merges into data/).')